In [10]:
# --- Notebook Setup ---
import numpy as np
import scipy.io
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import time
import os

# Important: Import TCN
try:
    from tcn import TCN, tcn_full_summary
except ImportError:
    print("ERROR: 'keras-tcn' library not found.")
    print("Please install it using: pip install keras-tcn")
    # Or handle the absence appropriately
    TCN = None

# Important: Import SHAP
try:
    import shap
except ImportError:
    print("WARNING: 'shap' library not found. XAI section will be skipped.")
    print("Install using: pip install shap")
    shap = None # Set to None if not found

# Ensure plots appear inline in the notebook
%matplotlib inline

# --- Configuration (Adjust as needed) ---
DATA_FILE_PATH = 's4_full.mat' # <<<--- IMPORTANT: Replace path
SEQUENCE_LENGTH = 59
TRAIN_TRIALS = [0, 1, 2]
TEST_TRIALS = [3, 4]
VALIDATION_SPLIT_RATIO = 0.2
EPOCHS = 20 # Adjusted epochs potentially needed for advanced model
BATCH_SIZE = 64
# --- Advanced TCN Model Hyperparameters ---
TCN_FILTERS = 96       # Increased filters
TCN_KERNEL_SIZE = 3
TCN_DILATIONS = [1, 2, 4, 8, 16] # Increased receptive field
TCN_NB_STACKS = 2      # Increased stacks
TCN_DROPOUT = 0.1
L2_REG = 1e-5          # L2 regularization factor for Dense layers

In [2]:
# --- Data Loading ---
def load_mat_data(file_path):
    """Loads data from .mat file."""
    print(f'Loading data from: {file_path}')
    # ... (rest of function remains the same as before) ...
    if not os.path.exists(file_path): print(f"Error: File not found {file_path}"); return None
    try:
        mat_data = scipy.io.loadmat(file_path)
        print('Data loaded.');
        if 'dsfilt_emg' not in mat_data or 'joint_angles' not in mat_data: print('Error: Required vars missing'); print(f"Keys: {list(mat_data.keys())}"); return None
        return mat_data
    except Exception as e: print(f'Error loading: {e}'); return None

# --- Data Splitting by Trial ---
def process_and_split_by_trial(mat_data, train_trials, test_trials):
    """Separates data based on trial index before concatenation."""
    print('Processing and splitting data by trial...')
    # ... (rest of function remains the same as before, including checks) ...
    try: dsfilt_emg, joint_angles = mat_data['dsfilt_emg'], mat_data['joint_angles']
    except KeyError: print("Error accessing keys"); return None
    if dsfilt_emg.shape[:2] != joint_angles.shape[:2]: print(f"Warn: Shape mismatch {dsfilt_emg.shape[:2]} vs {joint_angles.shape[:2]}")
    num_trials, num_tasks = dsfilt_emg.shape[0], dsfilt_emg.shape[1]; num_emg_channels, num_angles = -1, -1
    train_emg_list, train_angles_list, test_emg_list, test_angles_list = [], [], [], []
    for i in range(num_trials):
        for j in range(num_tasks):
            try: emg_segment, angle_segment = dsfilt_emg[i, j], joint_angles[i, j]
            except IndexError: continue
            if not isinstance(emg_segment, np.ndarray) or emg_segment.size == 0 or not isinstance(angle_segment, np.ndarray) or angle_segment.size == 0: continue
            if num_emg_channels == -1:
                if emg_segment.ndim == 2 and angle_segment.ndim == 2: num_emg_channels, num_angles = emg_segment.shape[1], angle_segment.shape[1]; print(f'Detected {num_emg_channels} EMG channels, {num_angles} angles.')
                else: print(f"Error: Bad dims ({i},{j})"); return None
            if emg_segment.ndim!=2 or angle_segment.ndim!=2 or emg_segment.shape[1]!=num_emg_channels or angle_segment.shape[1]!=num_angles: continue
            min_length = min(emg_segment.shape[0], angle_segment.shape[0])
            if min_length <= 0: continue
            emg_valid, angle_valid = emg_segment[:min_length, :], angle_segment[:min_length, :]
            if np.any(np.isnan(emg_valid)) or np.any(np.isinf(emg_valid)) or np.any(np.isnan(angle_valid)) or np.any(np.isinf(angle_valid)): continue
            if i in train_trials: train_emg_list.append(emg_valid); train_angles_list.append(angle_valid)
            elif i in test_trials: test_emg_list.append(emg_valid); test_angles_list.append(angle_valid)
    if not train_emg_list: print("Error: No valid train data"); return None
    try: train_emg_all, train_angles_all = np.concatenate(train_emg_list, axis=0), np.concatenate(train_angles_list, axis=0)
    except ValueError as e: print(f"Error concat train: {e}"); return None
    test_emg_all, test_angles_all = np.array([]), np.array([])
    if test_emg_list:
        try: test_emg_all, test_angles_all = np.concatenate(test_emg_list, axis=0), np.concatenate(test_angles_list, axis=0)
        except ValueError as e: print(f"Warn: Error concat test: {e}")
    print(f'Train Samples: {train_emg_all.shape[0]}, Test Samples: {test_emg_all.shape[0]}')
    return train_emg_all, train_angles_all, test_emg_all, test_angles_all, num_emg_channels, num_angles


# --- Normalization ---
def normalize_data(train_emg, train_angles, test_emg, test_angles):
    """Fits scaler on training data and transforms both train and test data."""
    print("Normalizing data (fitting scaler on train)...")
    # ... (rest of function remains the same as before) ...
    emg_scaler, angle_scaler = StandardScaler(), StandardScaler()
    try: train_emg_norm, train_angles_norm = emg_scaler.fit_transform(train_emg), angle_scaler.fit_transform(train_angles)
    except Exception as e: print(f"Error fit/transform train: {e}"); return None, None, None, None, None, None
    test_emg_norm, test_angles_norm = np.array([]), np.array([])
    if test_emg.size > 0:
        try: test_emg_norm, test_angles_norm = emg_scaler.transform(test_emg), angle_scaler.transform(test_angles)
        except Exception as e: print(f"Warn: Error transform test: {e}"); test_emg_norm, test_angles_norm = np.array([]), np.array([])
    print("Normalization complete."); return train_emg_norm, train_angles_norm, test_emg_norm, test_angles_norm, emg_scaler, angle_scaler

# --- Sequence Creation ---
def create_sequences(emg_data, angle_data, sequence_length, data_label="Data"):
    """Creates overlapping sequences for TCN input."""
    print(f"Creating sequences for {data_label}...")
    # ... (rest of function remains the same as before) ...
    if emg_data is None or angle_data is None or emg_data.size == 0 or angle_data.size == 0: print(f"Warn: Empty data for {data_label}"); return None, None
    num_samples = emg_data.shape[0]
    if num_samples < sequence_length: print(f'Warn: Not enough samples ({num_samples}) for {data_label} seq len {sequence_length}.'); return None, None
    try: num_features, num_targets = emg_data.shape[1], angle_data.shape[1]
    except IndexError: print(f"Error: Data {data_label} not 2D"); return None, None
    num_sequences = num_samples - sequence_length + 1
    try: X = np.zeros((num_sequences, sequence_length, num_features), dtype=np.float32); Y = np.zeros((num_sequences, num_targets), dtype=np.float32)
    except MemoryError: print(f"Error: Memory alloc failed for {data_label}"); return None, None
    for i in range(num_sequences): X[i] = emg_data[i : i + sequence_length, :]; Y[i] = angle_data[i + sequence_length - 1, :]
    print(f"Created {X.shape[0]} sequences for {data_label}."); return X, Y

In [3]:
# --- Advanced TCN Model Building ---
def build_advanced_tcn_model(sequence_length, num_features, num_targets,
                             nb_filters=TCN_FILTERS, kernel_size=TCN_KERNEL_SIZE,
                             dilations=TCN_DILATIONS, nb_stacks=TCN_NB_STACKS,
                             dropout_rate=TCN_DROPOUT, l2_reg=L2_REG):
    """Builds an enhanced TCN model with LayerNorm and L2 regularization."""
    print("Building Advanced TCN model architecture...")
    if TCN is None: raise ImportError("keras-tcn library is required but not found.")

    inputs = layers.Input(shape=(sequence_length, num_features), name='input_emg_sequence')

    # TCN Block
    x = TCN(nb_filters=nb_filters, kernel_size=kernel_size, nb_stacks=nb_stacks,
            dilations=dilations, padding='causal', use_skip_connections=True,
            dropout_rate=dropout_rate, activation=tf.keras.activations.relu,
            kernel_initializer='he_normal', return_sequences=False, # Get final output state
            name='tcn_block')(inputs)

    # Add Layer Normalization after TCN block
    x = layers.LayerNormalization(name='layer_norm_after_tcn')(x)

    # Dense layers with L2 regularization
    x = layers.Dense(nb_filters // 2, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2_reg),
                     name='dense_intermediate')(x)
    x = layers.Dropout(dropout_rate, name='dropout_final')(x)

    # Final output layer
    outputs = layers.Dense(num_targets, activation='linear', name='output_joint_angles')(x)

    # Create and Compile Model
    model = keras.Model(inputs=inputs, outputs=outputs, name="Advanced_TCN_Model")
    optimizer = keras.optimizers.Adam(learning_rate=0.001) # Consider AdamW for regularization
    model.compile(optimizer=optimizer,
                  loss='mean_squared_error',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    print("Advanced TCN Model built and compiled.")
    # Optional: Print detailed summary if keras-tcn supports it
    try: tcn_full_summary(model, expand_residual_blocks=False)
    except: model.summary(line_length=110)
    return model

In [4]:
# --- Model Training ---
def train_model(model, x_train, y_train, x_val, y_val, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """Trains the Keras model with early stopping."""
    print(f"\n--- Starting Model Training for {epochs} epochs ---")
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=12, # Increased patience slightly
        restore_best_weights=True, verbose=1
    )
    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val, y_val),
        callbacks=[early_stopping],
        verbose=1 # Show progress
    )
    print("Training complete.")
    # Plot training history
    plt.figure("Training History", figsize=(10, 4))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss (MSE)'); plt.title('Training and Validation Loss')
    plt.legend(); plt.grid(True)
    plt.tight_layout()
    # Note: plt.show() is usually not needed here in notebooks with %matplotlib inline
    return model, history # Return trained model and history

In [5]:
# --- Model Evaluation ---
def evaluate_model(model, x_test, y_test, angle_scaler):
    """Evaluates the model on the test set and returns predictions and metrics."""
    print("\n--- Evaluating Model on Test Set ---")
    if x_test is None or y_test is None:
        print("No test data available for evaluation.")
        return None, None, None, None # No results

    test_loss, test_rmse = model.evaluate(x_test, y_test, verbose=0)
    print(f"Test Loss (MSE): {test_loss:.4f}")
    print(f"Test RMSE (Scaled): {test_rmse:.4f}")

    print("Making predictions...")
    y_pred_scaled = model.predict(x_test)

    print("Inverse transforming predictions...")
    try:
        y_pred = angle_scaler.inverse_transform(y_pred_scaled)
        y_actual = angle_scaler.inverse_transform(y_test)
    except Exception as e:
        print(f"Error during inverse transform: {e}")
        return None, None, None, None

    # Calculate final metrics
    rmse_per_joint = np.sqrt(mean_squared_error(y_actual, y_pred, multioutput='raw_values'))
    r2_per_joint = r2_score(y_actual, y_pred, multioutput='raw_values')
    overall_rmse = np.mean(rmse_per_joint)
    variance_per_joint = np.var(y_actual, axis=0)
    valid_r2_indices = variance_per_joint > 1e-9
    overall_r2 = np.mean(r2_per_joint[valid_r2_indices]) if np.any(valid_r2_indices) else float('nan')

    print("\nFinal Evaluation Metrics (Inverse Transformed):")
    print(f"  Overall RMSE: {overall_rmse:.4f}")
    print(f"  Overall R-squared (avg): {overall_r2:.4f}")

    results = {
        'y_actual': y_actual,
        'y_pred': y_pred,
        'rmse_per_joint': rmse_per_joint,
        'r2_per_joint': r2_per_joint,
        'overall_rmse': overall_rmse,
        'overall_r2': overall_r2
    }
    return results

# --- Plotting Predictions ---
def plot_predictions(y_actual, y_pred, n_angles, num_to_plot=500, title_suffix=""):
    """Plots actual vs predicted joint angles for a subset of data."""
    print("Plotting actual vs. predicted angles...")
    joint_names = [f"Joint {i+1}" for i in range(n_angles)]
    plot_count = min(num_to_plot, y_actual.shape[0])
    plot_indices = range(plot_count)
    time_vector = np.array(plot_indices)

    # Select joints to plot (e.g., first 3 or specific ones)
    joints_to_plot_indices = [idx for idx in [0, 3, 6] if idx < n_angles]
    if not joints_to_plot_indices: joints_to_plot_indices = range(min(3, n_angles))

    plt.figure(f"Predicted vs Actual{title_suffix}", figsize=(12, 2 * len(joints_to_plot_indices)))
    plt.suptitle(f'Test Set: Actual vs Predicted Angles{title_suffix} (Sample)', y=1.02)
    for i, joint_idx in enumerate(joints_to_plot_indices):
        plt.subplot(len(joints_to_plot_indices), 1, i + 1)
        plt.plot(time_vector, y_actual[plot_indices, joint_idx], 'b-', label='Actual')
        plt.plot(time_vector, y_pred[plot_indices, joint_idx], 'r--', label='Predicted')
        plt.ylabel('Angle (deg)'); plt.title(joint_names[joint_idx])
        plt.legend(loc='upper right'); plt.grid(True)
        if i == len(joints_to_plot_indices) - 1: plt.xlabel('Sample Index')
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])

# --- Plotting Metrics ---
def plot_metrics(rmse_per_joint, r2_per_joint, n_angles, title_suffix=""):
    """Plots RMSE and R-squared per joint."""
    print("Plotting evaluation metrics per joint...")
    joint_names = [f"Joint {i+1}" for i in range(n_angles)]
    plt.figure(f"Performance Metrics{title_suffix}", figsize=(10, 8))
    plt.suptitle(f'Test Set Evaluation Metrics{title_suffix}', y=1.02)
    x_joints = np.arange(n_angles)
    # RMSE
    plt.subplot(2, 1, 1); plt.bar(x_joints, rmse_per_joint, color='skyblue'); plt.ylabel('RMSE (deg)'); plt.title('Test RMSE per Joint')
    plt.xticks(x_joints, joint_names, rotation=45, ha='right'); plt.grid(True, axis='y'); plt.xlim([-0.5, n_angles - 0.5])
    # R2
    plt.subplot(2, 1, 2); plt.bar(x_joints, r2_per_joint, color='lightcoral'); plt.ylabel('R-squared'); plt.title('Test R^2 per Joint')
    plt.xticks(x_joints, joint_names, rotation=45, ha='right')
    min_r2_val = np.nanmin(r2_per_joint) if not np.all(np.isnan(r2_per_joint)) else -0.1; plt.ylim([min(-0.1, min_r2_val - 0.1), 1.1]); plt.axhline(0, color='grey', lw=0.8, ls='--'); plt.grid(True, axis='y'); plt.xlim([-0.5, n_angles - 0.5])
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])

In [6]:
# --- SHAP Explanation Function ---
def explain_with_shap(model, x_train, x_test, n_angles, sequence_length, n_channels,
                      angle_scaler, # Needed for showing prediction value
                      background_size=100, explain_size=5,
                      sample_idx=0, output_idx=0, title_suffix=""):
    """Calculates and visualizes SHAP values for a given sample and output."""
    print(f"\n--- Explaining Prediction using SHAP (Sample {sample_idx+1}, Angle {output_idx+1}) ---")
    if shap is None: print("SHAP library not available. Skipping."); return
    if x_test is None: print("No test data to explain. Skipping."); return
    if sample_idx >= x_test.shape[0] or output_idx >= n_angles: print("Sample or output index out of bounds."); return

    try:
        # 1. Prepare data
        background_samples = min(background_size, x_train.shape[0])
        background_indices = np.random.choice(x_train.shape[0], background_samples, replace=False)
        background_data = x_train[background_indices]
        data_to_explain_subset = x_test[sample_idx:sample_idx+1] # Explain only the selected sample

        # 2. Create Explainer and Calculate SHAP (for the single sample)
        print(f"Using {background_data.shape[0]} samples as SHAP background.")
        explainer = shap.GradientExplainer(model, background_data)
        print(f"Calculating SHAP values for test sample index {sample_idx}...")
        # shap_values will be a list (one element per output)
        # Each element has shape (1, sequence_length, n_channels) for single sample explanation
        shap_values_list = explainer.shap_values(data_to_explain_subset)
        print("SHAP values calculated.")

        # 3. Extract values for the specific output and check
        shap_values_for_output = shap_values_list[output_idx][0] # Shape (sequence_length, n_channels)

        print(f"--- Checking SHAP values for plot (Shape: {shap_values_for_output.shape}) ---")
        if np.isnan(shap_values_for_output).all(): print("WARNING: All SHAP values are NaN!")
        elif np.all(shap_values_for_output == 0): print("WARNING: All SHAP values are zero!")
        else: print(f"SHAP Value Range: min={np.nanmin(shap_values_for_output):.4f}, max={np.nanmax(shap_values_for_output):.4f}, mean={np.nanmean(shap_values_for_output):.4f}")

        # 4. Get Prediction and Visualize Heatmap
        prediction_scaled = model.predict(data_to_explain_subset)[0] # Prediction for the sample
        prediction_actual_scale = angle_scaler.inverse_transform(prediction_scaled.reshape(1, -1))[0]
        predicted_value = prediction_actual_scale[output_idx]

        plt.figure(f"SHAP Values: Sample {sample_idx+1}, Angle {output_idx+1}{title_suffix}", figsize=(13, 5.5))
        img = plt.imshow(shap_values_for_output.T, cmap='coolwarm', aspect='auto', interpolation='nearest') # Channels (y) vs Time (x)
        plt.colorbar(img, label='SHAP Value (Contribution to Output)')
        plt.xlabel(f"Time Step (Sequence Length {sequence_length})"); plt.ylabel(f"EMG Channel (1 to {n_channels})")
        plt.yticks(ticks=np.arange(n_channels), labels=np.arange(1, n_channels + 1))
        plt.title(f"SHAP: Sample {sample_idx+1}, Pred Angle {output_idx+1} ({predicted_value:.2f} deg){title_suffix}")
        plt.figtext(0.5, 0.01, "Red: Pushes prediction higher. Blue: Pushes prediction lower.", ha="center", fontsize=9, style='italic')
        plt.tight_layout(rect=[0, 0.05, 1, 0.95]) # Adjust layout

        # Optional: Add waterfall plot for the same prediction (summarizes feature importance)
        # Note: Need to flatten SHAP values and create feature names for waterfall
        try:
            shap_values_flat = shap_values_for_output.flatten()
            # Create meaningful feature names (e.g., 'Ch1_T1', 'Ch1_T2', ...) - can be many!
            feature_names = [f'Ch{c+1}_T{t+1}' for t in range(sequence_length) for c in range(n_channels)]
            # Select top N features for clarity in waterfall plot
            num_waterfall_features = 15
            # Get the base value (expected value) from the explainer
            base_value = explainer.expected_value[output_idx]
            # Create SHAP explanation object
            exp = shap.Explanation(values=shap_values_flat,
                                   base_values=base_value,
                                   data=data_to_explain_subset[0].flatten(), # Flattened input data
                                   feature_names=feature_names)

            plt.figure(f"SHAP Waterfall: Sample {sample_idx+1}, Angle {output_idx+1}{title_suffix}", figsize=(8, 6))
            shap.plots.waterfall(exp, max_display=num_waterfall_features, show=False) # show=False as plt.show() isn't needed per plot
            plt.title(f"SHAP Waterfall: Top {num_waterfall_features} contributors\nSample {sample_idx+1}, Pred Angle {output_idx+1}")
            plt.tight_layout()
        except Exception as wf_err:
             print(f"Could not generate SHAP waterfall plot: {wf_err}")


    except Exception as e:
        print(f"\nError during SHAP explanation: {e}")
        import traceback
        traceback.print_exc() # Print detailed traceback for debugging

In [11]:
# --- Workflow ---
# 1. Load Data
raw_mat_data = load_mat_data(DATA_FILE_PATH)
if raw_mat_data is None: raise ValueError("Failed to load data.")

# 2. Process & Split
processed_data = process_and_split_by_trial(raw_mat_data, TRAIN_TRIALS, TEST_TRIALS)
if processed_data is None: raise ValueError("Failed to process data.")
tr_emg, tr_angles, test_emg, test_angles, n_channels, n_angles = processed_data
if n_channels <= 0 or n_angles <= 0: raise ValueError("Invalid data dimensions detected.")

# 3. Normalize
norm_data = normalize_data(tr_emg, tr_angles, test_emg, test_angles)
if norm_data[0] is None: raise ValueError("Failed normalization.")
tr_emg_norm, tr_angles_norm, test_emg_norm, test_angles_norm, emg_scaler, angle_scaler = norm_data

# 4. Create Sequences
x_train_val_seq, y_train_val_seq = create_sequences(tr_emg_norm, tr_angles_norm, SEQUENCE_LENGTH, "Train/Val Pool")
if x_train_val_seq is None: raise ValueError("Failed to create train sequences.")
x_test_seq, y_test_seq = create_sequences(test_emg_norm, test_angles_norm, SEQUENCE_LENGTH, "Test Pool")
if x_test_seq is None: print("Warning: No test sequences created."); x_test_seq, y_test_seq = np.array([]), np.array([])

# 5. Split Train/Validation
if x_train_val_seq.shape[0] < 2: raise ValueError("Not enough sequences for train/val split.")
x_train, x_val, y_train, y_val = train_test_split(x_train_val_seq, y_train_val_seq, test_size=VALIDATION_SPLIT_RATIO, random_state=42, shuffle=True)
x_test = x_test_seq if x_test_seq.size > 0 else None
y_test = y_test_seq if y_test_seq.size > 0 else None
print(f"Shapes: x_train={x_train.shape}, y_train={y_train.shape}, x_val={x_val.shape}, x_test={x_test.shape if x_test is not None else 'None'}")

# 6. Build Model
model = build_advanced_tcn_model(SEQUENCE_LENGTH, n_channels, n_angles)

# 7. Train Model
model, history = train_model(model, x_train, y_train, x_val, y_val, epochs=EPOCHS, batch_size=BATCH_SIZE)

# 8. Evaluate Model
eval_results = evaluate_model(model, x_test, y_test, angle_scaler)

# 9. Plot Results (if evaluation was possible)
if eval_results:
    plot_predictions(eval_results['y_actual'], eval_results['y_pred'], n_angles, title_suffix=" (Advanced TCN)")
    plot_metrics(eval_results['rmse_per_joint'], eval_results['r2_per_joint'], n_angles, title_suffix=" (Advanced TCN)")

# 10. Explain a Prediction with SHAP (if possible)
if eval_results and x_test is not None:
    explain_with_shap(model, x_train, x_test, n_angles, SEQUENCE_LENGTH, n_channels, angle_scaler,
                      sample_idx=0, output_idx=0, title_suffix=" (Advanced TCN)") # Explain first sample, first angle
    # You can call explain_with_shap again for different samples/angles here
    # explain_with_shap(model, x_train, x_test, n_angles, SEQUENCE_LENGTH, n_channels, angle_scaler, sample_idx=1, output_idx=1)

# Optional: Final plt.show() if some plots didn't appear automatically, but usually not needed with %matplotlib inline
# plt.show()

print("\n--- Workflow Complete ---")

Loading data from: s4_full.mat
Data loaded.
Processing and splitting data by trial...
Detected 8 EMG channels, 14 angles.
Train Samples: 84000, Test Samples: 56000
Normalizing data (fitting scaler on train)...
Normalization complete.
Creating sequences for Train/Val Pool...
Created 83942 sequences for Train/Val Pool.
Creating sequences for Test Pool...
Created 55942 sequences for Test Pool.
Shapes: x_train=(67153, 59, 8), y_train=(67153, 14), x_val=(16789, 59, 8), x_test=(55942, 59, 8)
Building Advanced TCN model architecture...
Advanced TCN Model built and compiled.


Model: "Advanced_TCN_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_emg_sequence (InputLayer) │ (None, 59, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tcn_block (TCN)                 │ (None, 96)             │       530,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_norm_after_tcn            │ (None, 96)             │           192 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_intermediate (Dense)      │ (None, 48)             │         4,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_final (Dropout)         │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_joint_angles (Dense)     │ (None, 14)             │           686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 535,934 (2.04 MB)

 Trainable params: 535,934 (2.04 MB)

 Non-trainable params: 0 (0.00 B)


--- Starting Model Training for 20 epochs ---
Epoch 1/20
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 96s 82ms/step - loss: 1.0058 - rmse: 1.0024 - val_loss: 1.0108 - val_rmse: 1.0051
Epoch 2/20
 824/1050 ━━━━━━━━━━━━━━━━━━━━ 18s 81ms/step - loss: 1.0069 - rmse: 1.0032

KeyboardInterrupt: 